# 1. 语音分类

## 1.1. 下载模型（可选：相关的数据集也可以下载）

- 下载模型
    - 本地目录：`C:\Users\ThinkPad\.cache\huggingface\hub`

- 数据集下载：
    - 使用transformers提供工具块datasets
    - 使用pytorch提供的数据集工具
        - torchaudio.datasets.SPEECHCOMMANDS 

In [1]:
import torchaudio
# 下载并加载数据集
ds = torchaudio.datasets.SPEECHCOMMANDS(
    root="/Users/logicye/Code/Datasets/",
    download=True
)
ds[0]

/Users/logicye/Code/ai_learning/learing_venv/lib/python3.13/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


(tensor([[-0.0658, -0.0709, -0.0753,  ..., -0.0700, -0.0731, -0.0704]]),
 16000,
 'backward',
 '0165e0e8',
 0)

## 1.2. 通过pipeline使用，并且掌握AudioClassificationPipeline类

In [1]:
import torchaudio
# 强制用 soundfile 读取，彻底绕开 ffmpeg
torchaudio.set_audio_backend("soundfile")

from transformers import pipeline
import torch

# 自动适配 M2 芯片（用 mps 加速）
device = "mps" if torch.backends.mps.is_available() else "cpu"

# 加载语音分类模型
pipe = pipeline(
    task="audio-classification",
    model="MIT/ast-finetuned-speech-commands-v2",
    device=device
)

# 你的音频路径
audio_path = "/Users/logicye/Code/Datasets/SpeechCommands/speech_commands_v0.02/backward/0165e0e8_nohash_0.wav"

# 开始识别
result = pipe(audio_path, top_k=1)

# 输出结果
print("="*50)
print(f"预测结果：{result[0]['label']}")
print(f"置信度：{result[0]['score']:.4f}")
print("="*50)

/var/folders/mb/f_7rzxgd24j1s22_0c4kt23m0000gn/T/ipykernel_17594/1349715879.py:3: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  torchaudio.set_audio_backend("soundfile")
/Users/logicye/Code/ai_learning/learing_venv/lib/python3.13/site-packages/torchaudio/_internal/module_utils.py:71: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  return func(*args, **kwargs)
/Users/logicye/Code/ai_learning/learing_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidget

预测结果：backward
置信度：1.0000


In [2]:
print(type(pipe.model))  # 了解我们使用的具体的模型
print(type(pipe.model.config))   # 适当做修改
print(type(pipe.processor))  # 多模态
print(type(pipe.tokenizer))  # 自然语言
print(type(pipe.image_processor))  # 机器视觉
print(type(pipe.feature_extractor))  # 语音识别
# print(pipe.model.config.id2label)
# print(pipe.model.config)
print(type(pipe.model.base_model))    # 特征模型
pipe.model.classifier

<class 'transformers.models.audio_spectrogram_transformer.modeling_audio_spectrogram_transformer.ASTForAudioClassification'>
<class 'transformers.models.audio_spectrogram_transformer.configuration_audio_spectrogram_transformer.ASTConfig'>
<class 'NoneType'>
<class 'NoneType'>
<class 'NoneType'>
<class 'transformers.models.audio_spectrogram_transformer.feature_extraction_audio_spectrogram_transformer.ASTFeatureExtractor'>
<class 'transformers.models.audio_spectrogram_transformer.modeling_audio_spectrogram_transformer.ASTModel'>


ASTMLPHead(
  (layernorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
  (dense): Linear(in_features=768, out_features=35, bias=True)
)

- 代码说明：
    - 在机器视觉使用feature_extractor是可以的，提示警告：不推荐使用，在未来的版本会删除该使用方式。
        - 逐步分开：
            - 机器视觉：image_processor
            - 自然语言：tokebbnizer
            - 多模态：processor(image_processor,tokenizer,feature_extractor三者的混合体)
            - 语音：feature_extractor独留给语音。

## 1.3. 直接使用模型与处理器